# 08 — Thesis Results Figures and Tables

This notebook reads the saved outputs from the benchmark notebooks and generates
dataset-level thesis/report-ready tables and figures.

## Inputs expected

All inputs are read from the structured result folders:

`results/<dataset>/<task>/<signal>/<eval_mode>/<control_mode>/tables/...`

From notebook `02_within_subject_benchmark.py`:
- `subject_summary_across_seeds.csv`
- `model_summary_across_subjects.csv`

From notebook `03_controls_permutation_and_prestim.py`:
- control `model_summary_across_subjects.csv`

From notebook `04_cross_subject_loso.py`:
- `loso_subject_summary_across_seeds.csv`
- `loso_model_summary_across_subjects.csv`

## Outputs produced

All generated report artifacts are saved under the selected experiment's artifact folder:

`results/<dataset>/<task>/<signal>/within_subject/standard/artifacts/thesis_results/`

### Tables
- Table 6.1 — grouped within-subject model summary
- Table 6.2 — main benchmark vs controls
- Table 6.3 — LOSO model summary

### Figures
- Figure 6.1 — grouped benchmark ROC-AUC by subject
- Figure 6.2 — grouped benchmark model ranking
- Figure 6.3 — main benchmark vs controls
- Figure 6.4 — LOSO model ranking

In [ ]:
# Run from the repository root so that relative result paths resolve
# identically regardless of where this notebook lives.
from oddball_eeg.paths import use_project_root

PROJECT_ROOT = use_project_root()

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from oddball_eeg.config import get_experiment_family
from oddball_eeg.io import save_dataframe, save_experiment_metadata, save_json
from oddball_eeg.visualization import (
    round_numeric_columns,
    save_figure,
    set_pandas_display_options,
)


set_pandas_display_options()

plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.size"] = 11

In [ ]:
def save_table_latex(
        df: pd.DataFrame,
        path: Path,
        decimals: int = 4,
        index: bool = False,
):
    out = round_numeric_columns(df.copy(), decimals=decimals)
    latex = out.to_latex(index=index, escape=False)
    path.write_text(latex, encoding="utf-8")

## 1. Resolve experiment directories

In [ ]:
# Change only this line when switching datasets.
EXPERIMENT_NAME = "ds005863_within_subject_main"
# EXPERIMENT_NAME = "ds003061_within_subject_main"

experiment_family = get_experiment_family(EXPERIMENT_NAME)

cfg_main = experiment_family["main"]
cfg_perm = experiment_family["permuted"]
cfg_prestim = experiment_family["prestim"]
cfg_loso = experiment_family["loso"]

MAIN_SUMMARY_DIR = cfg_main.table_dir("summaries")
PERM_SUMMARY_DIR = cfg_perm.table_dir("summaries")
PRESTIM_SUMMARY_DIR = cfg_prestim.table_dir("summaries")
LOSO_SUMMARY_DIR = cfg_loso.table_dir("summaries")

OUT_DIR = cfg_main.artifact_dir("thesis_results")
FIG_DIR = cfg_main.artifact_dir("thesis_results", "figures")
TAB_DIR = cfg_main.artifact_dir("thesis_results", "tables")

save_experiment_metadata(
    cfg_main,
    extra={
        "notebook": "08_thesis_results_figures_tables.ipynb",
        "purpose": "Generate thesis/report-ready result tables and figures from saved benchmark outputs.",
        "experiment_key": EXPERIMENT_NAME,
        "dataset": cfg_main.dataset.dataset,
        "task": cfg_main.dataset.task,
        "main_summary_dir": str(MAIN_SUMMARY_DIR),
        "permuted_summary_dir": str(PERM_SUMMARY_DIR),
        "prestim_summary_dir": str(PRESTIM_SUMMARY_DIR),
        "loso_summary_dir": str(LOSO_SUMMARY_DIR),
        "artifact_dir": str(OUT_DIR),
        "figure_dir": str(FIG_DIR),
        "table_dir": str(TAB_DIR),
    },
)

print("Experiment key:", EXPERIMENT_NAME)
print("Dataset       :", cfg_main.dataset.dataset)
print("Task          :", cfg_main.dataset.task)
print("Signal config :", cfg_main.signal)

print("\nInput summary directories:")
print("Main benchmark :", MAIN_SUMMARY_DIR)
print("Permuted       :", PERM_SUMMARY_DIR)
print("Pre-stimulus   :", PRESTIM_SUMMARY_DIR)
print("LOSO           :", LOSO_SUMMARY_DIR)

print("\nOutput directories:")
print("Artifacts:", OUT_DIR)
print("Figures  :", FIG_DIR)
print("Tables   :", TAB_DIR)

## 2. Load saved benchmark summaries

In [ ]:
# Main benchmark
df_main_subject = pd.read_csv(
    MAIN_SUMMARY_DIR / "subject_summary_across_seeds.csv"
)
df_main_model = pd.read_csv(
    MAIN_SUMMARY_DIR / "model_summary_across_subjects.csv"
)

# Controls
df_perm_model = pd.read_csv(
    PERM_SUMMARY_DIR / "model_summary_across_subjects.csv"
)
df_prestim_model = pd.read_csv(
    PRESTIM_SUMMARY_DIR / "model_summary_across_subjects.csv"
)

# LOSO
df_loso_subject = pd.read_csv(
    LOSO_SUMMARY_DIR / "loso_subject_summary_across_seeds.csv"
)
df_loso_model = pd.read_csv(
    LOSO_SUMMARY_DIR / "loso_model_summary_across_subjects.csv"
)

print("Main subject summary :", df_main_subject.shape)
print("Main model summary   :", df_main_model.shape)
print("Perm model summary   :", df_perm_model.shape)
print("Prestim model summary:", df_prestim_model.shape)
print("LOSO subject summary :", df_loso_subject.shape)
print("LOSO model summary   :", df_loso_model.shape)

## 3. Normalize model ordering

In [ ]:
MODEL_ORDER = ["lda", "logreg", "eegnet"]

MODEL_LABELS = {
    "lda": "LDA",
    "logreg": "Logistic Regression",
    "eegnet": "EEGNet",
}

MODEL_DISPLAY_ORDER = ["LDA", "Logistic Regression", "EEGNet"]


def normalize_model_key(x: str) -> str:
    s = str(x).strip().lower()
    s = s.replace("-", " ").replace("_", " ")
    s = " ".join(s.split())

    if s == "lda":
        return "lda"
    if s in {"logreg", "logistic regression", "logisticregression", "logistic", "lr"}:
        return "logreg"
    if s in {"eegnet", "eeg net"}:
        return "eegnet"
    return s


def apply_model_labels(df: pd.DataFrame, col: str = "model") -> pd.DataFrame:
    out = df.copy()
    out[col] = out[col].apply(normalize_model_key)
    out["model_key"] = out[col]
    out[col] = out[col].map(MODEL_LABELS).fillna(out[col])
    return out


df_main_subject = apply_model_labels(df_main_subject)
df_main_model = apply_model_labels(df_main_model)
df_perm_model = apply_model_labels(df_perm_model)
df_prestim_model = apply_model_labels(df_prestim_model)
df_loso_subject = apply_model_labels(df_loso_subject)
df_loso_model = apply_model_labels(df_loso_model)

## 4. Build Table 6.1 — grouped within-subject summary

In [ ]:
table_6_1 = (
    df_main_model[[
        "model",
        "mean_subject_auc",
        "std_subject_auc",
        "median_subject_auc",
        "worst_subject_auc",
    ]]
    .rename(columns={
        "model": "Model",
        "mean_subject_auc": "Mean AUC",
        "std_subject_auc": "Std AUC",
        "median_subject_auc": "Median AUC",
        "worst_subject_auc": "Worst AUC",
    })
    .copy()
)

table_6_1["order_key"] = table_6_1["Model"].map(
    {m: i for i, m in enumerate(MODEL_DISPLAY_ORDER)}
)
table_6_1 = table_6_1.sort_values("order_key").drop(columns="order_key").reset_index(drop=True)

display(round_numeric_columns(table_6_1, decimals=4))

## 5. Build Table 6.2 — main benchmark vs controls

In [ ]:
table_6_2 = (
    df_main_model[["model", "mean_subject_auc"]]
    .rename(columns={"mean_subject_auc": "Main benchmark"})
    .merge(
        df_perm_model[["model", "mean_subject_auc"]].rename(columns={"mean_subject_auc": "Permuted labels"}),
        on="model",
        how="left",
    )
    .merge(
        df_prestim_model[["model", "mean_subject_auc"]].rename(columns={"mean_subject_auc": "Pre-stimulus-only"}),
        on="model",
        how="left",
    )
    .rename(columns={"model": "Model"})
)

table_6_2["order_key"] = table_6_2["Model"].map(
    {m: i for i, m in enumerate(MODEL_DISPLAY_ORDER)}
)
table_6_2 = table_6_2.sort_values("order_key").drop(columns="order_key").reset_index(drop=True)

display(round_numeric_columns(table_6_2, decimals=4))

## 6. Build Table 6.3 — LOSO summary

In [ ]:
table_6_3 = (
    df_loso_model[[
        "model",
        "mean_subject_auc",
        "std_subject_auc",
        "median_subject_auc",
        "worst_subject_auc",
    ]]
    .rename(columns={
        "model": "Model",
        "mean_subject_auc": "Mean AUC",
        "std_subject_auc": "Std AUC",
        "median_subject_auc": "Median AUC",
        "worst_subject_auc": "Worst AUC",
    })
    .copy()
)

table_6_3["order_key"] = table_6_3["Model"].map(
    {m: i for i, m in enumerate(MODEL_DISPLAY_ORDER)}
)
table_6_3 = table_6_3.sort_values("order_key").drop(columns="order_key").reset_index(drop=True)

display(round_numeric_columns(table_6_3, decimals=4))

## 7. Save the thesis tables

In [ ]:
save_dataframe(
    table_6_1,
    TAB_DIR / "table_6_1_grouped_summary.csv",
)

save_dataframe(
    table_6_2,
    TAB_DIR / "table_6_2_control_results.csv",
)

save_dataframe(
    table_6_3,
    TAB_DIR / "table_6_3_loso_summary.csv",
)

save_table_latex(table_6_1, TAB_DIR / "table_6_1_grouped_summary.tex", decimals=4, index=False)
save_table_latex(table_6_2, TAB_DIR / "table_6_2_control_results.tex", decimals=4, index=False)
save_table_latex(table_6_3, TAB_DIR / "table_6_3_loso_summary.tex", decimals=4, index=False)

print("Saved tables to:", TAB_DIR.resolve())
display(round_numeric_columns(table_6_1, 4))
display(round_numeric_columns(table_6_2, 4))
display(round_numeric_columns(table_6_3, 4))

## 8. Print LaTeX rows for direct thesis insertion

In [ ]:
def df_to_latex_rows(df, decimals=4):
    out = round_numeric_columns(df, decimals=decimals).copy()
    rows = []
    for _, row in out.iterrows():
        rows.append(" & ".join(map(str, row.values)) + r" \\")
    return "\n".join(rows)

print("Table 6.1 rows:\n")
print(df_to_latex_rows(table_6_1))

print("\nTable 6.2 rows:\n")
print(df_to_latex_rows(table_6_2))

print("\nTable 6.3 rows:\n")
print(df_to_latex_rows(table_6_3))

## 9. Figure 6.1 — grouped within-subject ROC-AUC by subject

In [ ]:
plot_df = df_main_subject[[
    "subject",
    "model",
    "auc_seed_mean",
    "auc_seed_std",
]].copy()

plot_df["subject"] = plot_df["subject"].astype(str).str.strip().str.zfill(3)

plot_df = plot_df.rename(columns={
    "auc_seed_mean": "auc_mean",
    "auc_seed_std": "auc_std",
})

plot_df["model_display"] = plot_df["model"]

print("Model counts:")
print(plot_df["model_display"].value_counts(dropna=False))

print("\nMapped model counts:")
print(plot_df["model_display"].value_counts(dropna=False))

unmapped = plot_df.loc[plot_df["model_display"].isna(), "model"].unique()
print("\nUnmapped model names:", unmapped)

subject_order = sorted(plot_df["subject"].unique().tolist())
model_order = MODEL_DISPLAY_ORDER

x = np.arange(len(subject_order))
width = 0.8 / len(model_order)

fig, ax = plt.subplots(figsize=(10, 5))

for i, model in enumerate(model_order):
    sub_df = (
        plot_df[plot_df["model_display"] == model]
        .set_index("subject")
        .reindex(subject_order)
    )

    ax.bar(
        x + i * width - (len(model_order) - 1) * width / 2,
        sub_df["auc_mean"].values,
        width=width,
        yerr=sub_df["auc_std"].values,
        capsize=3,
        label=model,
    )

ax.set_xticks(x)
ax.set_xticklabels(subject_order)
ax.set_xlabel("Subject")
ax.set_ylabel("ROC-AUC")
ax.set_ylim(0.0, 1.0)
ax.set_title(
    f"Grouped within-subject benchmark performance by subject\n"
    f"{cfg_main.dataset.dataset}, {cfg_main.dataset.task}"
)
ax.legend()
ax.grid(axis="y", alpha=0.3)

fig.tight_layout()
save_figure(fig, FIG_DIR / "figure_6_1_grouped_by_subject.png")
plt.show()

## 10. Figure 6.2 — grouped benchmark model ranking

In [ ]:
plot_df = (
    df_main_model[["model", "mean_subject_auc", "std_subject_auc"]]
    .copy()
    .rename(columns={
        "model": "Model",
        "mean_subject_auc": "Mean AUC",
        "std_subject_auc": "Std AUC",
    })
)

order = MODEL_DISPLAY_ORDER
plot_df["order_key"] = plot_df["Model"].map({m: i for i, m in enumerate(order)})
plot_df = plot_df.sort_values("order_key").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(plot_df["Model"], plot_df["Mean AUC"], yerr=plot_df["Std AUC"], capsize=4)

for i, v in enumerate(plot_df["Mean AUC"]):
    ax.text(i, v + 0.015, f"{v:.4f}", ha="center")

ax.set_ylim(0.0, 1.0)
ax.set_ylabel("Mean subject ROC-AUC")
ax.set_title(
    f"Grouped within-subject model ranking\n"
    f"{cfg_main.dataset.dataset}, {cfg_main.dataset.task}"
)
ax.grid(axis="y", alpha=0.3)

fig.tight_layout()
save_figure(fig, FIG_DIR / "figure_6_2_grouped_model_summary.png")
plt.show()

## 11. Figure 6.3 — main benchmark vs controls

In [ ]:
plot_df = table_6_2.copy()

x = np.arange(len(plot_df))
w = 0.25

fig, ax = plt.subplots(figsize=(8, 4.8))

ax.bar(x - w, plot_df["Main benchmark"], width=w, label="Main benchmark")
ax.bar(x, plot_df["Permuted labels"], width=w, label="Permuted labels")
ax.bar(x + w, plot_df["Pre-stimulus-only"], width=w, label="Pre-stimulus-only")

ax.set_xticks(x)
ax.set_xticklabels(plot_df["Model"])
ax.set_ylim(0.0, 1.0)
ax.set_ylabel("Mean subject ROC-AUC")
ax.set_title(
    f"Main grouped benchmark vs anti-overfitting controls\n"
    f"{cfg_main.dataset.dataset}, {cfg_main.dataset.task}"
)
ax.legend()
ax.grid(axis="y", alpha=0.3)

fig.tight_layout()
save_figure(fig, FIG_DIR / "figure_6_3_controls.png")
plt.show()

## 12. Figure 6.4 — LOSO model ranking

In [ ]:
plot_df = (
    df_loso_model[["model", "mean_subject_auc", "std_subject_auc"]]
    .copy()
    .rename(columns={
        "model": "Model",
        "mean_subject_auc": "Mean AUC",
        "std_subject_auc": "Std AUC",
    })
)

order = MODEL_DISPLAY_ORDER
plot_df["order_key"] = plot_df["Model"].map({m: i for i, m in enumerate(order)})
plot_df = plot_df.sort_values("order_key").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(plot_df["Model"], plot_df["Mean AUC"], yerr=plot_df["Std AUC"], capsize=4)

for i, v in enumerate(plot_df["Mean AUC"]):
    ax.text(i, v + 0.015, f"{v:.4f}", ha="center")

ax.set_ylim(0.0, 1.0)
ax.set_ylabel("Mean held-out-subject ROC-AUC")
ax.set_title(
    f"LOSO model ranking\n"
    f"{cfg_loso.dataset.dataset}, {cfg_loso.dataset.task}"
)
ax.grid(axis="y", alpha=0.3)

fig.tight_layout()
save_figure(fig, FIG_DIR / "figure_6_4_loso_summary.png")
plt.show()

## 13. Optional comparison figure — within-subject vs LOSO by subject
This one is optional for the dissertation, but useful for analysis.

In [ ]:
main_vs_loso = (
    df_main_subject[["subject", "model", "auc_seed_mean"]]
    .rename(columns={"auc_seed_mean": "within_subject_auc"})
    .merge(
        df_loso_subject[["subject", "model", "auc_seed_mean"]].rename(columns={"auc_seed_mean": "loso_auc"}),
        on=["subject", "model"],
        how="inner",
    )
)

main_vs_loso["delta"] = (
        main_vs_loso["within_subject_auc"] - main_vs_loso["loso_auc"]
)

main_vs_loso = round_numeric_columns(main_vs_loso, decimals=4)

display(main_vs_loso.head(12))

save_dataframe(
    main_vs_loso,
    TAB_DIR / "optional_within_subject_vs_loso_by_subject.csv",
)

## 14. Save a small manifest

In [ ]:
manifest = {
    "experiment_key": EXPERIMENT_NAME,
    "dataset": cfg_main.dataset.dataset,
    "task": cfg_main.dataset.task,
    "signal": cfg_main.signal.slug(),
    "artifact_dir": str(OUT_DIR),
    "table_dir": str(TAB_DIR),
    "figure_dir": str(FIG_DIR),
    "tables_csv": sorted([p.name for p in TAB_DIR.glob("*.csv")]),
    "tables_tex": sorted([p.name for p in TAB_DIR.glob("*.tex")]),
    "figures_png": sorted([p.name for p in FIG_DIR.glob("*.png")]),
}

save_json(
    manifest,
    OUT_DIR / "manifest.json",
)

save_experiment_metadata(
    cfg_main,
    extra={
        "notebook": "08_thesis_results_figures_tables.ipynb",
        "status": "completed",
        "experiment_key": EXPERIMENT_NAME,
        "artifact_manifest": manifest,
    },
)

manifest

## 15. Chapter 6 note

If the values generated here differ slightly from the prose draft, trust the regenerated CSV-based values and update the dissertation text to match the actual saved benchmark outputs.